In [ ]:
import os
import string

import cv2
import emoji
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from PIL import Image
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [ ]:
# embedder = SentenceTransformer(
#     "intfloat/multilingual-e5-large",
# )


# def encode_titles(titles, batch_size=64):
#     titles = ["query: " + str(t) for t in titles]
#     emb = embedder.encode(
#         titles,
#         batch_size=batch_size,
#         show_progress_bar=True,
#         normalize_embeddings=True,
#     )
#     return emb

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
# clip = model.to(device)
# clip.eval()


# def extract_clip_image_embeddings(image_dir, video_ids, batch_size=32):
#     embeddings = []
#     missing = []
#     for i in tqdm(range(0, len(video_ids), batch_size)):
#         batch_ids = video_ids[i : i + batch_size]
#         images = []
#         for vid in batch_ids:
#             path = os.path.join(image_dir, f"{vid}.jpg")
#             try:
#                 img = Image.open(path).convert("RGB")
#             except:
#                 img = Image.new("RGB", (224, 224), color=0)
#                 missing.append(vid)
#             images.append(img)
#         inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
#         with torch.no_grad():
#             feats = clip.get_image_features(**inputs)
#             feats = feats / feats.norm(dim=1, keepdim=True)
#         embeddings.append(feats.cpu().numpy())
#     embeddings = np.vstack(embeddings)
#     return embeddings, missing


In [ ]:
pca = PCA(n_components=50, random_state=42)

train_emb = np.load("embeddings/train_emb.npy")
test_emb = np.load("embeddings/test_emb.npy")

train_pca = pca.fit_transform(train_emb)
test_pca = pca.transform(test_emb)

train_img_emb = np.load("embeddings/clip_image_train.npy")
test_img_emb = np.load("embeddings/clip_image_test.npy")

train_img_emb_pca = pca.fit_transform(train_img_emb)
test_img_emb_pca = pca.transform(test_img_emb)

qwen_train = np.load("embeddings/qwen_train.npy")
qwen_test = np.load("embeddings/qwen_test.npy")

qwen_train_pca = pca.fit_transform(qwen_train)
qwen_test_pca = pca.transform(qwen_test)

In [ ]:
def extract_thumbnail_features(img_path):
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img)
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    brightness = hsv[:, :, 2].mean()
    saturation = hsv[:, :, 1].mean()
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    edge_density = edges.mean() / 255.0
    return {
        "brightness": brightness,
        "saturation": saturation,
        "edge_density": edge_density,
    }


def build_thumbnail_df(df, img_dir):
    rows = []
    for vid in df["video_id"]:
        img_path = os.path.join(img_dir, f"{vid}.jpg")
        if not os.path.exists(img_path):
            rows.append({"brightness": 0.0, "saturation": 0.0, "edge_density": 0.0})
            continue
        feats = extract_thumbnail_features(img_path)
        rows.append(feats)
    return pd.DataFrame(rows, index=df.index)

In [ ]:
analyzer = SentimentIntensityAnalyzer()


def make_features(df, title_emb, img_emb, qwen_emb, img_dir):
    """Create base features (without target encoding)"""

    t_ref = pd.to_datetime(df["published_at"], utc=True).max()
    t = pd.to_datetime(df["published_at"], utc=True)
    title = df["title"].fillna("")
    hashtags = df["title"].str.findall(r"#\w+")
    X_thumbnail = build_thumbnail_df(df, img_dir)
    X = pd.DataFrame(index=df.index)
    X["log_subs"] = np.log1p(df["subscriber_count"])
    X["log_duration"] = np.log1p(df["duration"])
    X["log_days_since_publish"] = np.log1p((t_ref - t).dt.days)
    X["channel_activity"] = X["log_subs"] * X["log_days_since_publish"]
    X["num_hashtag"] = hashtags.apply(len)
    X["num_emoji"] = df["title"].apply(
        lambda x: len([c for c in str(x) if c in emoji.EMOJI_DATA])
    )
    X["is_weekend"] = (t.dt.dayofweek >= 5).astype(int)
    X["year"] = t.dt.year
    X["pub_weekday"] = t.dt.weekday
    X["pub_hour"] = t.dt.hour
    X["hour_sin"] = np.sin(2 * np.pi * t.dt.hour / 24)
    X["hour_cos"] = np.cos(2 * np.pi * t.dt.hour / 24)
    X["weekday_sin"] = np.sin(2 * np.pi * t.dt.day / 7)
    X["weekday_cos"] = np.cos(2 * np.pi * t.dt.day / 7)
    X["month_cos"] = np.cos(2 * np.pi * t.dt.month / 12)
    X["month_sin"] = np.sin(2 * np.pi * t.dt.month / 12)
    X["title_word_count"] = df["title"].apply(lambda x: len(str(x).split()))
    X["num_exclamation"] = title.str.count("!")
    X["num_question"] = title.str.count(r"\?")
    X["num_ellipsis"] = title.str.count(r"\.\.\.")
    X["num_punct"] = title.apply(lambda s: sum(1 for c in s if c in string.punctuation))
    X["num_all_caps_words"] = (
        df["title"]
        .fillna("")
        .apply(lambda s: sum(1 for w in s.split() if w.isupper() and len(w) > 1))
    )
    X["title_sentiment"] = df["title"].apply(
        lambda x: analyzer.polarity_scores(str(x))["compound"]
    )
    X["sub_tier"] = pd.qcut(
        df["subscriber_count"], q=10, labels=False, duplicates="drop"
    )
    X["duration_bin"] = pd.qcut(df["duration"], q=10, labels=False, duplicates="drop")
    X["days_bin"] = pd.qcut((t_ref - t).dt.days, q=10, labels=False, duplicates="drop")
    X_title = pd.DataFrame(
        title_emb,
        index=df.index,
        columns=[f"title_emb_{i}" for i in range(title_emb.shape[1])],
    )
    X_qwen = pd.DataFrame(
        qwen_emb,
        index=df.index,
        columns=[f"qwen_emb_{i}" for i in range(qwen_emb.shape[1])],
    )
    X_img = pd.DataFrame(
        img_emb,
        index=df.index,
        columns=[f"img_emb_{i}" for i in range(img_emb.shape[1])],
    )
    X = pd.concat([X, X_title, X_img, X_qwen, X_thumbnail], axis=1)
    return X


def add_target_encoding(X_tr, X_val, X_te, y_tr, categorical_cols):
    """
    Add target encoding features for categorical columns.
    Computes mean, std, count on training fold only.
    Args:
        X_tr, X_val, X_te: Train/validation/test dataframes
        y_tr: Training targets
        categorical_cols: List of column names to encode
    Returns:
        Modified X_tr, X_val, X_te with TE features added and categorical cols dropped
    """
    for col in categorical_cols:
        stats_df = pd.DataFrame({col: X_tr[col], "target": y_tr})
        agg_stats = (
            stats_df.groupby(col)["target"].agg(["mean", "std", "count"]).reset_index()
        )
        mean_map = agg_stats.set_index(col)["mean"]
        std_map = agg_stats.set_index(col)["std"]
        count_map = agg_stats.set_index(col)["count"]
        for df_ in [X_tr, X_val, X_te]:
            df_[f"{col}_te_mean"] = df_[col].map(mean_map).fillna(y_tr.mean())
            df_[f"{col}_te_std"] = df_[col].map(std_map).fillna(y_tr.std())
            df_[f"{col}_te_count"] = df_[col].map(count_map).fillna(0)
    X_tr.drop(categorical_cols, axis=1, inplace=True)
    X_val.drop(categorical_cols, axis=1, inplace=True)
    X_te.drop(categorical_cols, axis=1, inplace=True)
    return X_tr, X_val, X_te

In [ ]:
X_train = make_features(
    train, train_pca, qwen_train_pca, train_img_emb_pca, "thumbnail/train"
)
y = train["virality_score"]
X_test = make_features(
    test, test_pca, qwen_test_pca, test_img_emb_pca, "thumbnail/test"
)

In [ ]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train))
test_pred_lgb = np.zeros(len(X_test))
categorical_cols = ["sub_tier", "duration_bin", "days_bin"]
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"Fold {fold + 1}/{N_SPLITS}")
    X_tr = X_train.iloc[tr_idx].copy()
    X_val = X_train.iloc[val_idx].copy()
    X_te = X_test.copy()
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]
    X_tr, X_val, X_te = add_target_encoding(X_tr, X_val, X_te, y_tr, categorical_cols)
    feature_names = X_tr.columns
    model = LGBMRegressor(
        n_estimators=2500,
        learning_rate=0.01,
        max_depth=-1,
        num_leaves=100,
        subsample=0.85,
        colsample_bytree=0.70,
        objective="regression",
        random_state=42,
        verbose=1,
    )
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100),
        ],
    )
    oof_lgb[val_idx] = model.predict(X_val)
    test_pred_lgb += model.predict(X_te) / N_SPLITS
    fold_rmse = root_mean_squared_error(y_val, oof_lgb[val_idx])
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.4f}")
cv_rmse = root_mean_squared_error(y, oof_lgb)
print(f"Final CV RMSE: {cv_rmse:.4f}")

In [ ]:
imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(
    ascending=False
)
top10 = imp.head(20)
plt.figure(figsize=(8, 8))
plt.barh(top10.index[::-1], top10.values[::-1])
plt.xlabel("Feature Importance")
plt.title("Feature Importance (Top 20)")
plt.tight_layout()
plt.show()

In [ ]:
oof_xgb = np.zeros(len(X_train))
test_pred_xgb = np.zeros(len(X_test))


for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"Fold {fold + 1}/{N_SPLITS}")
    X_tr = X_train.iloc[tr_idx].copy()
    X_val = X_train.iloc[val_idx].copy()
    X_te = X_test.copy()
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]
    X_tr, X_val, X_te = add_target_encoding(X_tr, X_val, X_te, y_tr, categorical_cols)
    feature_names = X_tr.columns
    X_tr_arr = X_tr.values
    X_val_arr = X_val.values
    X_te_arr = X_te.values

    model = xgb.XGBRegressor(
        n_estimators=2500,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.7,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        early_stopping_rounds=100,
        n_jobs=-1,
    )

    model.fit(X_tr_arr, y_tr, eval_set=[(X_val_arr, y_val)], verbose=100)
    oof_xgb[val_idx] = model.predict(X_val_arr)
    test_pred_xgb += model.predict(X_te_arr) / N_SPLITS
    fold_rmse = root_mean_squared_error(y_val, oof_xgb[val_idx])
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.4f}")

    
cv_rmse = root_mean_squared_error(y, oof_xgb)
print(f"Final CV RMSE: {cv_rmse:.4f}")

In [ ]:
simp = pd.Series(model.feature_importances_, index=feature_names).sort_values(
    ascending=False
)
top10 = imp.head(30)
plt.figure(figsize=(8, 8))
plt.barh(top10.index[::-1], top10.values[::-1])
plt.xlabel("Feature Importance")
plt.title("Top 10 XGBoost Features")
plt.tight_layout()
plt.show()

In [ ]:
oof_cat = np.zeros(len(X_train))
test_pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"Fold {fold + 1}/{N_SPLITS}")
    X_tr = X_train.iloc[tr_idx].copy()
    X_val = X_train.iloc[val_idx].copy()
    X_te = X_test.copy()
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]
    X_tr, X_val, X_te = add_target_encoding(X_tr, X_val, X_te, y_tr, categorical_cols)
    feature_names = X_tr.columns
    model = CatBoostRegressor(
        iterations=4000,
        learning_rate=0.01,
        depth=6,
        subsample=0.8,
        colsample_bylevel=0.8,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_state=42,
        early_stopping_rounds=100,
        verbose=100,
        thread_count=-1,
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = model.predict(X_val)
    test_pred_cat += model.predict(X_te) / N_SPLITS
    fold_rmse = root_mean_squared_error(y_val, oof_cat[val_idx])
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.4f}")

    
cv_rmse = root_mean_squared_error(y, oof_cat)
print(f"Final CV RMSE: {cv_rmse:.4f}")

In [ ]:
from sklearn.linear_model import Ridge

X_meta_train = np.column_stack([oof_lgb, oof_xgb, oof_cat])
X_meta_test = np.column_stack([test_pred_lgb, test_pred_xgb, test_pred_cat])
ridge = Ridge(alpha=1.0)
ridge.fit(X_meta_train, y)
oof_stack = ridge.predict(X_meta_train)
rmse = root_mean_squared_error(y, oof_stack)
print("Stacked CV RMSE:", rmse)

In [ ]:
print(ridge.coef_)
test_stack = ridge.predict(X_meta_test)
test_pred = pd.DataFrame({"video_id": test["video_id"], "virality_score": test_stack})
test_pred.to_csv("submission.csv", index=False)

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("estimators", 1000, 10000, log=True),
        "learning_rate": trial.suggest_float("lr", 0.005, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("l1", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("l2", 1e-8, 10.0, log=True),
        "objective": "regression",
        "random_state": 42,
        "verbosity": -1,
    }
    rmses = []
    for tr_idx, val_idx in kf.split(X_train):
        X_tr = X_train.iloc[tr_idx].copy()
        X_val = X_train.iloc[val_idx].copy()
        X_te = X_test.copy()
        y_tr = y.iloc[tr_idx]
        y_val = y.iloc[val_idx]
        X_tr, X_val, X_te = add_target_encoding(
            X_tr, X_val, X_te, y_tr, categorical_cols
        )
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(100)],
        )
        preds = model.predict(X_val)
        rmses.append(root_mean_squared_error(y_val, preds))
    return np.mean(rmses)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)